In [1]:
!pip install --no-index --find-links=/kaggle/input/ariel-2024-pqdm pqdm
!pip install catboost

Looking in links: /kaggle/input/ariel-2024-pqdm
Processing /kaggle/input/ariel-2024-pqdm/pqdm-0.2.0-py2.py3-none-any.whl
Processing /kaggle/input/ariel-2024-pqdm/bounded_pool_executor-0.0.3-py3-none-any.whl (from pqdm)


In [2]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
from pqdm.threads import pqdm
from astropy.stats import sigma_clip
from scipy.signal import savgol_filter
from catboost import CatBoostRegressor, Pool
import pickle

ROOT_PATH = "/kaggle/input/ariel-data-challenge-2025"
MODE = "test"
__t0 = time.perf_counter()

# =========================================================
# CONFIGURATION
# =========================================================

class Config:
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"
    SCALE = 0.946
    SIGMA = 0.00056
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 11
    N_JOBS = 3
    
    # CatBoost parameters
    CATBOOST_ITERATIONS = 1000
    CATBOOST_DEPTH = 6
    CATBOOST_LR = 0.03

# =========================================================
# PREPROCESSING MODULE
# =========================================================

class SignalProcessor:
    """Handles all data preprocessing and calibration"""
    
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(
            f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', 
            index_col='planet_id'
        ).index.astype(int)

    def _apply_linear_corr(self, linear_corr, signal):
        """Apply linearity correction to signal"""
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]
        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        """Calibrate raw signal for a single planet and sensor"""
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        
        # Load raw data
        signal = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet"
        ).to_numpy()
        dark = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet"
        ).to_numpy()
        dead = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet"
        ).to_numpy()
        flat = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet"
        ).to_numpy()
        linear_corr = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet"
        ).values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])
        
        # Reshape and apply ADC corrections
        signal = signal.reshape(sensor_cfg["raw_shape"])
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset
        
        # Detect hot pixels
        hot = sigma_clip(dark, sigma=5, maxiters=5).mask
        
        # Apply sensor-specific cropping
        if sensor == "AIRS-CH0":
            signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
        
        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark = dark[y0:y1, x0:x1]
            dead = dead[y0:y1, x0:x1]
            flat = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot = hot[y0:y1, x0:x1]
        
        # Apply corrections
        np.maximum(signal, 0, out=signal)
        
        # Apply linearity correction
        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)
        
        # Dark subtraction with pattern
        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale = base_dt + increment
        signal[::2] -= dark * even_scale
        signal[1::2] -= dark * odd_scale
        
        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor):
        """Preprocess calibrated signal with binning and filtering"""
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]
        
        # Extract ROI
        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        # CDS (Correlated Double Sampling)
        mean_signal = np.nanmean(signal_roi, axis=1)
        cds_signal = mean_signal[1::2] - mean_signal[0::2]
        
        # Binning
        n_bins = cds_signal.shape[0] // binning
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])
        
        # Outlier clipping for AIRS-CH0
        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)
            np.clip(binned, q_lo, q_hi, out=binned)
        
        # Reshape for FGS1
        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))
        
        # Variance-based weighting for AIRS-CH0
        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)
            med = np.nanmedian(var)
            safe_var = np.where(
                ~np.isfinite(var) | (var <= 0), 
                med if (np.isfinite(med) and med > 0) else 1.0, 
                var
            )
            w = 1.0 / safe_var
            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)
            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)
            binned *= w[None, :]
        
        return binned

    def _process_planet_sensor(self, args):
        """Process a single planet-sensor combination"""
        planet_id, sensor = args['planet_id'], args['sensor']
        calibrated = self._calibrate_single_signal(planet_id, sensor)
        preprocessed = self._preprocess_calibrated_signal(calibrated, sensor)
        return preprocessed

    def process_all_data(self):
        """Process all planets and sensors in parallel"""
        print("[PREPROCESSING] Processing FGS1 data...")
        args_fgs1 = [
            dict(planet_id=planet_id, sensor="FGS1") 
            for planet_id in self.planet_ids
        ]
        preprocessed_fgs1 = pqdm(
            args_fgs1, 
            self._process_planet_sensor, 
            n_jobs=self.cfg.N_JOBS
        )
        
        print("[PREPROCESSING] Processing AIRS-CH0 data...")
        args_airs_ch0 = [
            dict(planet_id=planet_id, sensor="AIRS-CH0") 
            for planet_id in self.planet_ids
        ]
        preprocessed_airs_ch0 = pqdm(
            args_airs_ch0, 
            self._process_planet_sensor, 
            n_jobs=self.cfg.N_JOBS
        )
        
        # Combine FGS1 and AIRS-CH0
        preprocessed_signal = np.concatenate([
            np.stack(preprocessed_fgs1), 
            np.stack(preprocessed_airs_ch0)
        ], axis=2)
        
        print(f"[PREPROCESSING] Complete. Shape: {preprocessed_signal.shape}")
        return preprocessed_signal

# =========================================================
# FEATURE ENGINEERING
# =========================================================

def extract_features(preprocessed_data, star_info_df):
    """Extract features from preprocessed data for CatBoost"""
    features_list = []
    
    for i, signal in enumerate(tqdm(preprocessed_data, desc="Extracting features")):
        # FGS1 features (column 0)
        fgs_signal = signal[:, 0]
        
        # AIRS features (columns 1+)
        airs_signal = signal[:, 1:].mean(axis=1)
        airs_signal_smooth = savgol_filter(airs_signal, 23, 2)
        
        # Statistical features
        features = {
            # FGS1 features
            'fgs_mean': np.nanmean(fgs_signal),
            'fgs_std': np.nanstd(fgs_signal),
            'fgs_min': np.nanmin(fgs_signal),
            'fgs_max': np.nanmax(fgs_signal),
            'fgs_median': np.nanmedian(fgs_signal),
            'fgs_range': np.nanmax(fgs_signal) - np.nanmin(fgs_signal),
            
            # AIRS features
            'airs_mean': np.nanmean(airs_signal),
            'airs_std': np.nanstd(airs_signal),
            'airs_min': np.nanmin(airs_signal),
            'airs_max': np.nanmax(airs_signal),
            'airs_median': np.nanmedian(airs_signal),
            'airs_range': np.nanmax(airs_signal) - np.nanmin(airs_signal),
            
            # Smoothed AIRS features
            'airs_smooth_min': np.nanmin(airs_signal_smooth),
            'airs_smooth_std': np.nanstd(airs_signal_smooth),
            
            # Transit-like features
            'transit_depth_estimate': np.nanmax(airs_signal) - np.nanmin(airs_signal_smooth),
            'min_position': np.argmin(airs_signal_smooth) / len(airs_signal_smooth),
            
            # Gradient features
            'airs_grad_mean': np.nanmean(np.abs(np.gradient(airs_signal_smooth))),
            'airs_grad_max': np.nanmax(np.abs(np.gradient(airs_signal_smooth))),
            
            # Spectral features (variance across wavelengths)
            'spectral_variance': np.nanvar(signal[:, 1:], axis=1).mean(),
        }
        
        features_list.append(features)
    
    features_df = pd.DataFrame(features_list)
    
    # Add stellar parameters
    if star_info_df is not None:
        features_df = pd.concat([features_df, star_info_df.reset_index(drop=True)], axis=1)
    
    return features_df

In [3]:
class CatBoostWavelengthPredictor:
    """CatBoost model for predicting wavelength-dependent transit depths"""
    
    def __init__(self, config, n_wavelengths=282):
        self.cfg = config
        self.n_wavelengths = n_wavelengths
        self.models = []
        
    def train(self, X_train, y_train, X_val=None, y_val=None):
        """Train separate CatBoost model for each wavelength"""
        print(f"[CATBOOST] Training {self.n_wavelengths} models...")
        
        for i in tqdm(range(self.n_wavelengths), desc="Training wavelengths"):
            model = CatBoostRegressor(
                iterations=self.cfg.CATBOOST_ITERATIONS,
                depth=self.cfg.CATBOOST_DEPTH,
                learning_rate=self.cfg.CATBOOST_LR,
                loss_function='RMSE',
                random_seed=42,
                verbose=False
            )
            
            if X_val is not None and y_val is not None:
                eval_set = Pool(X_val, y_val[:, i])
                model.fit(X_train, y_train[:, i], eval_set=eval_set, verbose=False)
            else:
                model.fit(X_train, y_train[:, i], verbose=False)
            
            self.models.append(model)
    
    def predict(self, X):
        """Predict transit depths for all wavelengths"""
        predictions = np.zeros((len(X), self.n_wavelengths))
        
        for i, model in enumerate(tqdm(self.models, desc="Predicting")):
            predictions[:, i] = model.predict(X)
        
        return predictions
    
    def save(self, filepath):
        """Save all models"""
        with open(filepath, 'wb') as f:
            pickle.dump(self.models, f)
        print(f"[CATBOOST] Models saved to {filepath}")
    
    def load(self, filepath):
        """Load all models"""
        with open(filepath, 'rb') as f:
            self.models = pickle.load(f)
        print(f"[CATBOOST] Models loaded from {filepath}")

In [4]:
config = Config()
    
# Step 1: Preprocessing
print("="*60)
print("STEP 1: PREPROCESSING")
print("="*60)
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()
    
# Save preprocessed data
np.save('preprocessed_data.npy', preprocessed_data)
print(f"[INFO] Preprocessed data saved. Shape: {preprocessed_data.shape}")
    
# Step 2: Load star info
print("\n" + "="*60)
print("STEP 2: LOADING STELLAR PARAMETERS")
print("="*60)
star_info = pd.read_csv(f"{ROOT_PATH}/{MODE}_star_info.csv")
star_info["planet_id"] = star_info["planet_id"].astype(int)
planet_ids = star_info["planet_id"].tolist()
star_info = star_info.set_index("planet_id")
    
# Step 3: Feature engineering
print("\n" + "="*60)
print("STEP 3: FEATURE ENGINEERING")
print("="*60)
features_df = extract_features(preprocessed_data, star_info)
print(f"[INFO] Features extracted. Shape: {features_df.shape}")
print(f"[INFO] Features: {list(features_df.columns)}")
    
# Step 4: CatBoost prediction
print("\n" + "="*60)
print("STEP 4: CATBOOST PREDICTION")
print("="*60)

STEP 1: PREPROCESSING
[PREPROCESSING] Processing FGS1 data...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

[PREPROCESSING] Processing AIRS-CH0 data...


...

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

[PREPROCESSING] Complete. Shape: (1, 187, 283)
[INFO] Preprocessed data saved. Shape: (1, 187, 283)

STEP 2: LOADING STELLAR PARAMETERS

STEP 3: FEATURE ENGINEERING


Extracting features: 100%|██████████| 1/1 [00:00<00:00, 13.16it/s]

[INFO] Features extracted. Shape: (1, 27)
[INFO] Features: ['fgs_mean', 'fgs_std', 'fgs_min', 'fgs_max', 'fgs_median', 'fgs_range', 'airs_mean', 'airs_std', 'airs_min', 'airs_max', 'airs_median', 'airs_range', 'airs_smooth_min', 'airs_smooth_std', 'transit_depth_estimate', 'min_position', 'airs_grad_mean', 'airs_grad_max', 'spectral_variance', 'Rs', 'Ms', 'Ts', 'Mp', 'e', 'P', 'sma', 'i']

STEP 4: CATBOOST PREDICTION


In [5]:
catboost_model = CatBoostWavelengthPredictor(config)

print("[WARNING] No pre-trained CatBoost models found. Using fallback method.")

predictions_df = pd.DataFrame({
        "planet_id": planet_ids, 
        "transit_depth": features_df['transit_depth_estimate'].values
    })
print("[INFO] Using feature-based predictions")
    
__t1 = time.perf_counter()
elapsed = __t1 - __t0
print(f"\n[TIMING] Total runtime: {elapsed:.2f} s ({elapsed/60:.2f} min)")
print("[INFO] Preprocessing and feature extraction complete!")
print("[INFO] Next step: Train CatBoost models or load pre-trained models for prediction")

[WARNING] No pre-trained CatBoost models found. Using fallback method.
[INFO] Using feature-based predictions

[TIMING] Total runtime: 7.94 s (0.13 min)
[INFO] Preprocessing and feature extraction complete!
[INFO] Next step: Train CatBoost models or load pre-trained models for prediction


In [6]:
predictions_df.to_csv(f"submission.csv", index=False)